In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FixedFormatter
from matplotlib.legend_handler import HandlerPatch
from scipy.stats import gaussian_kde, chi2
from matplotlib.colors import to_hex, to_rgb
from pathlib import Path

# =========================================================
# 0. Global settings
# =========================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "axes.unicode_minus": False,
    "axes.labelsize": 15,
    "axes.titlesize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 10.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# =========================================================
# 1. Data
# =========================================================
base_dir  = Path(r"E:\关中干旱\index")
data_path = base_dir / "final_indices_for_clustering.csv"
out_path  = base_dir / "final_combined_plot_refined.pdf"

df = pd.read_csv(data_path)
df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", regex=True)]

df["SOM_Cluster"] = (
    pd.to_numeric(df.get("SOM_Cluster", np.nan), errors="coerce")
    .fillna(0).astype(int)
)

# =========================================================
# 2. Factors
# =========================================================
factors = {
    "东亚槽":           "EAT",
    "南海副高面积":     "SCSH",
    "西太平洋副高北界": "WPSH",
    "北极涛动":         "AO",
    "冬季北极涛动":     "AO-W",
}
if "春季北极涛动" in df.columns:
    factors["春季北极涛动"] = "AO-S"

indices = [
    k for k in [
        "东亚槽", "南海副高面积", "西太平洋副高北界",
        "北极涛动", "冬季北极涛动", "春季北极涛动",
    ]
    if k in df.columns
]
if len(indices) == 0:
    raise ValueError("No expected factor columns found in the CSV file.")

n = len(indices)

# =========================================================
# 3. Cluster color system
# =========================================================
# 干旱年直接使用原色，正常/湿润年做 48% 白色 tint，边框做 65% 暗化
CLUSTER_BASE = {
    0: np.array(to_rgb("#FE2400")),   # 鲜红   ← Cluster I
    1: np.array(to_rgb("#64D000")),   # 鲜绿   ← Cluster II
    2: np.array(to_rgb("#833198")),   # 深紫   ← Cluster III
}

WHITE      = np.array([1.0, 1.0, 1.0])
TINT_RATIO = 0.48

def cluster_drought_color(cid):
    """干旱年：原色不变"""
    return to_hex(CLUSTER_BASE[cid])

def cluster_normal_color(cid):
    """正常/湿润年：48% 白色 tint"""
    tinted = CLUSTER_BASE[cid] * (1 - TINT_RATIO) + WHITE * TINT_RATIO
    return to_hex(np.clip(tinted, 0, 1))

def cluster_edge_color(cid):
    """边框：65% 暗化"""
    darkened = np.clip(CLUSTER_BASE[cid] * 0.65, 0, 1)
    return to_hex(darkened)

cluster_labels_roman = {
    0: "Cluster I",
    1: "Cluster II",
    2: "Cluster III",
}

# =========================================================
# 4. SPI-3 drought threshold
# =========================================================
spi_col = "spring_SPI3"
if spi_col not in df.columns:
    raise ValueError("Column 'spring_SPI3' not found in the CSV file.")

DROUGHT_THRESH = -1.0

df[spi_col] = pd.to_numeric(df[spi_col], errors="coerce")
df["SPI3_type"] = df[spi_col].apply(
    lambda v: "drought" if (pd.notna(v) and v < DROUGHT_THRESH) else "normal"
)

# =========================================================
# 5. Prepare 2025 row
# =========================================================
year_col = next((c for c in ["Year", "year", "年份", "年度"] if c in df.columns), None)
year_2025_row = None
year_2025_cid = None

if year_col is not None:
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce")
    mask_2025 = df[year_col] == 2025
    if mask_2025.any():
        year_2025_row = df.loc[mask_2025].iloc[0]
        year_2025_cid = int(year_2025_row["SOM_Cluster"])

# =========================================================
# 6. Visual tuning
# =========================================================
scatter_alpha = 0.88
ellipse_alpha = 0.85
ellipse_lw    = 1.8
KDE_SCALE     = 0.5

DATA_TICKS  = [-3, 0, 3]
DATA_LABELS = ["-3", "0", "3"]
DENS_TICKS  = [0, KDE_SCALE]
DENS_LABELS = ["0", "0.5"]

normal_s   = 24
drought_s  = 52
edge_lw    = 1.6
year2025_s = 130

# =========================================================
# 7. Custom legend handler for ellipse
# =========================================================
class HandlerEllipse(HandlerPatch):
    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        center = (xdescent + width / 2, ydescent + height / 2)
        p = Ellipse(
            xy=center,
            width=width * 0.95,
            height=height * 0.65,
            angle=0,
            fill=False,
            linestyle=orig_handle.get_linestyle(),
            linewidth=orig_handle.get_linewidth(),
            edgecolor=orig_handle.get_edgecolor(),
            alpha=orig_handle.get_alpha()
        )
        p.set_transform(trans)
        return [p]

# =========================================================
# 8. KDE helpers
# =========================================================
def _draw_kde_h(ax, data, color):
    data = pd.to_numeric(data, errors="coerce").dropna()
    if len(data) < 2:
        return
    kde = gaussian_kde(data)
    x = np.linspace(-3, 3, 300)
    d = kde(x)
    if d.max() > 0:
        d = d / d.max() * KDE_SCALE
    ax.plot(x, d, color=color, lw=2.2, alpha=0.92, ls="-", zorder=3)

def _draw_kde_v(ax, data, color):
    data = pd.to_numeric(data, errors="coerce").dropna()
    if len(data) < 2:
        return
    kde = gaussian_kde(data)
    y = np.linspace(-3, 3, 300)
    d = kde(y)
    if d.max() > 0:
        d = d / d.max() * KDE_SCALE
    ax.plot(d, y, color=color, lw=2.2, alpha=0.92, ls="-", zorder=3)

# =========================================================
# 9. Axis helpers
# =========================================================
def setup_kde_h(ax, show_xlabel=False, xlabel="",
                show_ylabel=False, ylabel=""):
    ax.set_xlim(-3, 3)
    ax.set_ylim(0, KDE_SCALE)
    ax.set_box_aspect(1)
    ax.xaxis.set_major_locator(FixedLocator(DATA_TICKS))
    ax.xaxis.set_major_formatter(FixedFormatter(DATA_LABELS))
    ax.yaxis.set_major_locator(FixedLocator(DENS_TICKS))
    ax.yaxis.set_major_formatter(FixedFormatter(DENS_LABELS))
    ax.grid(True, ls="--", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    if show_xlabel:
        ax.set_xlabel(xlabel, labelpad=6)
    if show_ylabel:
        ax.set_ylabel(ylabel, labelpad=6)

def setup_kde_v(ax, show_xlabel=False, xlabel="",
                show_ylabel=False, ylabel=""):
    ax.set_xlim(0, KDE_SCALE)
    ax.set_ylim(-3, 3)
    ax.set_box_aspect(1)
    ax.xaxis.set_major_locator(FixedLocator(DENS_TICKS))
    ax.xaxis.set_major_formatter(FixedFormatter(DENS_LABELS))
    ax.yaxis.set_major_locator(FixedLocator(DATA_TICKS))
    ax.yaxis.set_major_formatter(FixedFormatter(DATA_LABELS))
    ax.grid(True, ls="--", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    if show_xlabel:
        ax.set_xlabel(xlabel, labelpad=6)
    if show_ylabel:
        ax.set_ylabel(ylabel, labelpad=6)

# =========================================================
# 10. Ellipse
# =========================================================
def plot_ellipse(ax, data, x, y, color, level=0.9):
    d = data[[x, y]].dropna()
    if len(d) < 2:
        return
    cov  = np.cov(d[x], d[y])
    mean = d.mean()
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals, 0, None)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    w, h  = 2 * np.sqrt(eigvals * chi2.ppf(level, 2))
    ax.add_patch(
        Ellipse(
            (mean[x], mean[y]), w, h,
            angle=angle,
            fill=False,
            lw=ellipse_lw,
            ls="--",
            edgecolor=color,
            alpha=ellipse_alpha,
        )
    )

# =========================================================
# 11. Figure layout
# =========================================================
fig, axes = plt.subplots(
    n, n,
    figsize=(15, 15),
    gridspec_kw={
        "wspace": 0.38,
        "hspace": 0.38,
    },
)
if n == 1:
    axes = np.array([[axes]])

# =========================================================
# 12. Main plotting loop
# =========================================================
for i, ycol in enumerate(indices):
    for j, xcol in enumerate(indices):
        ax = axes[i, j]

        # 上三角隐藏
        if i < j:
            ax.axis("off")
            continue

        # 对角线 → KDE
        if i == j:
            use_h = (j != n - 1)
            for cid in [0, 1, 2]:
                d   = df[df["SOM_Cluster"] == cid][xcol]
                col = cluster_drought_color(cid)
                if use_h:
                    _draw_kde_h(ax, d, col)
                else:
                    _draw_kde_v(ax, d, col)

            if use_h:
                setup_kde_h(
                    ax,
                    show_xlabel=(i == n - 1),
                    xlabel=factors.get(xcol, xcol),
                    show_ylabel=(j == 0),
                    ylabel=factors.get(ycol, ycol),
                )
            else:
                setup_kde_v(
                    ax,
                    show_xlabel=(i == n - 1),
                    xlabel=factors.get(xcol, xcol),
                    show_ylabel=(j == 0),
                    ylabel=factors.get(ycol, ycol),
                )
            continue

        # 下三角 → 散点
        valid = (
            pd.to_numeric(df[xcol], errors="coerce").notna() &
            pd.to_numeric(df[ycol], errors="coerce").notna() &
            df[spi_col].notna()
        )
        dplot = df.loc[valid].copy()

        dplot_normal  = dplot[dplot["SPI3_type"] != "drought"]
        dplot_drought = dplot[dplot["SPI3_type"] == "drought"]

        # 正常/湿润年（tint 浅色，无边框，垫底）
        for cid in [0, 1, 2]:
            sub = dplot_normal[dplot_normal["SOM_Cluster"] == cid]
            if sub.empty:
                continue
            ax.scatter(
                sub[xcol], sub[ycol],
                c=cluster_normal_color(cid),
                s=normal_s,
                alpha=scatter_alpha,
                edgecolors="none",
                zorder=2,
            )

        # 干旱年（原色 + 加深边框，置顶）
        for cid in [0, 1, 2]:
            sub = dplot_drought[dplot_drought["SOM_Cluster"] == cid]
            if sub.empty:
                continue
            ax.scatter(
                sub[xcol], sub[ycol],
                c=cluster_drought_color(cid),
                s=drought_s,
                alpha=scatter_alpha,
                edgecolors=cluster_edge_color(cid),
                linewidths=edge_lw,
                zorder=4,
            )

        # 2025 星形
        if year_2025_row is not None:
            px = pd.to_numeric(year_2025_row[xcol], errors="coerce")
            py = pd.to_numeric(year_2025_row[ycol], errors="coerce")
            if np.isfinite(px) and np.isfinite(py):
                cid_s = year_2025_cid if year_2025_cid is not None else 0
                star_face = (
                    cluster_drought_color(cid_s)
                    if year_2025_row["SPI3_type"] == "drought"
                    else cluster_normal_color(cid_s)
                )
                ax.scatter([px], [py],
                           s=year2025_s * 1.6, marker="*",
                           facecolors="white",
                           edgecolors=cluster_edge_color(cid_s),
                           linewidths=1.9, zorder=6)
                ax.scatter([px], [py],
                           s=year2025_s, marker="*",
                           facecolors=star_face,
                           edgecolors=cluster_edge_color(cid_s),
                           linewidths=1.1, zorder=7)

        # 置信椭圆
        for cid in [0, 1, 2]:
            plot_ellipse(
                ax, df[df["SOM_Cluster"] == cid],
                xcol, ycol,
                cluster_drought_color(cid),
            )

        ax.axhline(0, ls="--", color="black", alpha=0.28)
        ax.axvline(0, ls="--", color="black", alpha=0.28)
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.xaxis.set_major_locator(FixedLocator(DATA_TICKS))
        ax.xaxis.set_major_formatter(FixedFormatter(DATA_LABELS))
        ax.yaxis.set_major_locator(FixedLocator(DATA_TICKS))
        ax.yaxis.set_major_formatter(FixedFormatter(DATA_LABELS))
        ax.set_box_aspect(1)
        ax.grid(True, ls="--", alpha=0.18)

        if i == n - 1:
            ax.set_xlabel(factors.get(xcol, xcol), labelpad=6)
        if j == 0:
            ax.set_ylabel(factors.get(ycol, ycol), labelpad=6)

# =========================================================
# 13. Legend
# =========================================================
cluster_patch_handles = []
for cid in [0, 1, 2]:
    h = Line2D(
        [0], [0],
        marker="s",
        linestyle="None",
        markerfacecolor=cluster_drought_color(cid),
        markeredgecolor=cluster_edge_color(cid),
        markeredgewidth=0.8,
        markersize=10,
        label=cluster_labels_roman[cid],
    )
    cluster_patch_handles.append(h)

drought_handle = Line2D(
    [0], [0], marker="o", linestyle="None",
    markerfacecolor=cluster_drought_color(0),
    markeredgecolor=cluster_edge_color(0),
    markeredgewidth=1.5,
    markersize=8,
    label=r"Drought year (SPI-3 $<$ $-$1.0)",
)
normal_handle = Line2D(
    [0], [0], marker="o", linestyle="None",
    markerfacecolor=cluster_normal_color(0),
    markeredgecolor="none",
    markersize=7,
    label="Normal / wet year",
)

kde_handle = Line2D(
    [0], [0], color="#555555", lw=2.0, ls="-",
    label="Kernel density estimate",
)
ellipse_handle = Ellipse(
    (0, 0), width=1.0, height=0.6,
    fill=False, edgecolor="#555555",
    linestyle="--", linewidth=1.8,
    alpha=0.90,
    label="90% confidence ellipse",
)

star_face_leg = (
    cluster_drought_color(year_2025_cid)
    if year_2025_cid is not None else "#888888"
)
star_edge_leg = (
    cluster_edge_color(year_2025_cid)
    if year_2025_cid is not None else "#444444"
)
star_handle = Line2D(
    [0], [0], marker="*", linestyle="None",
    markerfacecolor=star_face_leg,
    markeredgecolor=star_edge_leg,
    markeredgewidth=1.1,
    markersize=13,
    label="Year 2025",
)

def _spacer(label=""):
    return Line2D([0], [0], linestyle="None", marker="None", label=label)

all_handles = (
    [_spacer("Cluster identity")]
    + cluster_patch_handles
    + [_spacer("Year type")]
    + [drought_handle, normal_handle]
    + [_spacer("Plot elements")]
    + [kde_handle, ellipse_handle, star_handle]
)

leg = fig.legend(
    handles=all_handles,
    loc="upper right",
    bbox_to_anchor=(0.995, 0.980),
    frameon=True,
    framealpha=0.92,
    edgecolor="#CCCCCC",
    fancybox=False,
    ncol=1,
    handletextpad=0.9,
    labelspacing=0.55,
    borderpad=0.8,
    borderaxespad=0.5,
    handler_map={Ellipse: HandlerEllipse()},
)

leg_texts       = leg.get_texts()
leg_handles_obj = leg.legend_handles
title_indices   = [0, 4, 7]
for idx in title_indices:
    if idx < len(leg_texts):
        leg_texts[idx].set_fontweight("bold")
        leg_texts[idx].set_color("#444444")
        leg_texts[idx].set_fontsize(9.5)
    if idx < len(leg_handles_obj):
        leg_handles_obj[idx].set_visible(False)

plt.subplots_adjust(right=0.78)

# =========================================================
# 14. Save
# =========================================================
plt.savefig(out_path, format="pdf", bbox_inches="tight")
plt.close()
print(f"PDF saved: {out_path}")

# =========================================================
# 15. Cluster statistics
# =========================================================
print("\n" + "=" * 60)
print("  CLUSTER STATISTICS")
print("=" * 60)

year_col_stat = next(
    (c for c in ["Year", "year", "年份", "年度"] if c in df.columns), None
)

for cid in sorted(df["SOM_Cluster"].unique()):
    sub     = df[df["SOM_Cluster"] == cid].copy()
    sub_spi = sub[spi_col].dropna()

    n_total     = len(sub)
    n_drought   = int((sub["SPI3_type"] == "drought").sum())
    n_normal    = n_total - n_drought
    pct_drought = n_drought / n_total * 100 if n_total > 0 else 0.0

    print(f"\n── {cluster_labels_roman.get(cid, f'Cluster {cid}')} ──")
    print(f"  Total years      : {n_total}")
    print(f"  Drought years    : {n_drought}  ({pct_drought:.1f}%)")
    print(f"  Normal/wet years : {n_normal}  ({100 - pct_drought:.1f}%)")
    print(f"  SPI-3  mean      : {sub_spi.mean():+.4f}")
    print(f"  SPI-3  median    : {sub_spi.median():+.4f}")
    print(f"  SPI-3  std       : {sub_spi.std():.4f}")
    print(f"  SPI-3  min/max   : {sub_spi.min():+.4f} / {sub_spi.max():+.4f}")
    print(f"  Index means:")
    for col in indices:
        vals  = pd.to_numeric(sub[col], errors="coerce").dropna()
        label = factors.get(col, col)
        if len(vals) > 0:
            print(f"    {label:<8}: {vals.mean():+.4f}  (std={vals.std():.4f})")
    if year_col_stat is not None and n_drought > 0:
        drought_years = (
            sub.loc[sub["SPI3_type"] == "drought", year_col_stat]
            .dropna().astype(int).sort_values().tolist()
        )
        print(f"  Drought year list: {drought_years}")

print("\n" + "=" * 60)
print("\n── Global Summary ──")
total         = len(df)
total_drought = int((df["SPI3_type"] == "drought").sum())
print(f"  Total records    : {total}")
print(f"  Total drought    : {total_drought}  ({total_drought/total*100:.1f}%)")
print(f"  Overall SPI-3 mean : {df[spi_col].mean():+.4f}")
print(f"  Overall SPI-3 std  : {df[spi_col].std():.4f}")
if year_col_stat is not None:
    yr = df[year_col_stat].dropna()
    print(f"  Year range       : {int(yr.min())} – {int(yr.max())}")
print("=" * 60)
